# Scooter Number Estimation

Three methods can be considered:

1. Relying entirely on Company B's data
* Pros: No errors due to difference with between Company A
* Cons: There doesn't seem to be a viable method with the current data. For example, if location information were sufficiently precise, we could match the end location of earlier entries with start locations of later entries to create chains where each chain corresponds to one scooter. However, the current location resolution is around 1 mile, making it impossible to distinguish between different scooters.

2. Selecting a feature common to both Company A and Company B (e.g. number of rides, driven miles) that correlates with scooter numbers 
    With an example of using "number of rides" as the common feature,
   1. First get the following constant:
   num_ride_per_scooter_company_A = num_ride_company_A / num_scooter_company_A_operate
   
   2. Then calculate the number of scooters Company B operates:
   num_scooter_company_B_operate = num_ride_company_B / num_ride_per_scooter_company_A
   
   * Pros: Features are available with current data
   * Cons: 
        * Error due to any difference between Company A and Company B. 
        * Company A data doesn't cover Company B's January period. 
        * Most importantly, this method is incorrect if there's no feature that has a proportional relationship with num_scooter, and we've seen that increasing scooters didn't increase number of rides. 

3. Similar to method 2, but make assumptions about num_ride_per_scooter_company_B_assumed:
   num_scooter_company_B_operate = num_ride_company_B / num_ride_per_scooter_company_B_assumed
   
* Pros: 
    * Possible with current data. 
    * Can cover Company B's January period.
* Cons: 
    * If the constant value assumption changes over time, it will be incorrect. 
* The assumed value might be very different from reality. Therefore, we are using simple num_ride_per_scooter_company_B_assumed feature, which is easy enough to gauge by gut feeling.


In conclusion, method 1 and method 2 are discarded, and method 3 is chosen. 

* Assume a value for num_ride_per_scooter_company_B_assumed
   * Assumption 1: Use num_ride_per_scooter_company_A in December
   * Assumption 2: Use maximum num_ride_per_scooter_company_A (this value measured weekly basis)
   * Assumption 3: Use minimum num_ride_per_scooter_company_A (this value measured weekly basis)

The three results are shown in the final plot.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from preprocess_data import get_all_data_frames
import plotly.express as px


In [2]:
def get_num_scooter(df_week : pd.DataFrame):
    # week	company	num_ride	num_scooter
    df = df_week.groupby(['week','company'])[['num_ride', 'num_scooter']].sum().sort_values(by=['company', 'week'], ascending=True).reset_index()

    #                num_ride	num_scooter
    # company	     A	B	    A	B
    # week				
    # 2018-08-13	1727.0	NaN	205.0	NaN
    df = df.pivot(index = 'week', columns = 'company', values = ['num_ride', 'num_scooter'])

    # Get weekly ride_per_scooter value from Company A
    df['ride_per_scooter_a'] = df['num_ride']['A']/df['num_scooter']['A']

    ######### Alternative Methods
    assumed_val = {}

    # Estimate num_scooter for Company B operate each week - Use December Mean of ride_per_scooter of Company A 
    num_ride_per_scooter_per_week_december = df['ride_per_scooter_a'].dropna()[-4:].mean()
    df['assumption_1'] = df['num_ride']['B']/num_ride_per_scooter_per_week_december
    assumed_val['assumption_1'] = num_ride_per_scooter_per_week_december

    # Estimate num_scooter for Company B operate each week - Use max ride_per_scooter of Company A 
    num_ride_per_scooter_per_week_max = df['ride_per_scooter_a'].max()
    df['assumption_2'] = df['num_ride']['B']/num_ride_per_scooter_per_week_max
    assumed_val['assumption_2'] = num_ride_per_scooter_per_week_max

    # Estimate num_scooter for Company B operate each week - Use min ride_per_scooter of Company A 
    num_ride_per_scooter_per_week_min = df['ride_per_scooter_a'].min()
    df['assumption_3'] = df['num_ride']['B']/num_ride_per_scooter_per_week_min
    assumed_val['assumption_3'] = num_ride_per_scooter_per_week_min


    # Handle multi-index columns
    df = df[-9:][[
        'assumption_1',
        'assumption_2',
        'assumption_3',
        ]].reset_index()
    
    df.columns = df.columns.get_level_values(0)

    # melt to use hue/col like features for plotting
    df = pd.melt(
        df,
        col_level=0,
        id_vars=['week'],
        value_vars=[
            'assumption_1',
            'assumption_2',
            'assumption_3',
        ],
        var_name='num_ride_per_scooter_per_week',
        value_name='num_scooter_b_estimated'
    )

    return df, assumed_val

In [3]:
df_dict = get_all_data_frames() 
df_week = df_dict['df_week']

/Users/minjungkim/Developer/company_compare/preprocess_data.py:86: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df_b['week'] = df_b.start_time.dt.tz_convert('US/Eastern').dt.to_period('W').dt.start_time


In [4]:
df, assumed_val = get_num_scooter(df_week)
fig = px.line(df, x="week", y="num_scooter_b_estimated", color='num_ride_per_scooter_per_week',
              title='Estimated Number of Scooters Company B Operates')
# Change legend text
fig.update_traces(
    name=str(round(assumed_val['assumption_1'],1))+' (assumption_1)',  
    selector=dict(name='assumption_1')
).update_traces(
    name=str(round(assumed_val['assumption_2'],1))+' (assumption_2)',  
    selector=dict(name='assumption_2')
).update_traces(
    name=str(round(assumed_val['assumption_3'],1))+' (assumption_3)', 
    selector=dict(name='assumption_3')
)
fig.show()